# WO-06 — comparing a place's basin with the basins around it
Fills S2.3, S5.2, S6.2, S6.3, S6.4 and S6.5 from the ring: the basins that share a border with the place's basin. Wording is in `profile_templates.md`; the rules are in `scripts/edop/whg/wo06_rings.py`. Cell 2 fetches the local values of every basin in each ring once and saves them to `output/edop/whg/wo06_ring_values.csv` (long; resumable); everything after it runs on that file, so tuning a rule does not mean re-fetching. Reads the local server, the local database and the saved WO-03 ring and flow tables. Read-only except `wo06_filled_profiles.md`, `output/edop/whg/wo06_ring_values.csv` and `output/edop/whg/wo06_fill.csv`.

In [1]:
%matplotlib inline
# Cell 1 — Setup: wording file, server check, saved ring and flow tables
import sys, time, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C
from scripts.edop.whg import wo04_fill as W
from scripts.edop.whg import wo06_rings as R

conn = db_connect()
PAIRS = pd.read_csv(C.OUT / 'q1_pairs.csv', dtype={'tag': str})
FLOW = pd.read_csv(C.OUT / 'q2_flow_pairs.csv', dtype={'tag': str})
SAMPLE = C.population_points()
print('commit', C.commit(), '| server:', W.get_json('/api/health'), '| wording:', W.templates_stamp(), '| test places:', len(C.PLACES), '| sample points:', len(SAMPLE))

commit fd6050d | server: {'status': 'ok'} | wording: profile_templates.md v5 (last commit 58fae48; edited since: yes); script wo04_fill.py (last commit fd6050d; edited since: yes); script wo06_rings.py (last commit fd6050d; edited since: yes) | test places: 10 | sample points: 306


In [2]:
# Cell 2 — Fetch the ring values: the ten test places and the 306 sample basins (about 7 s per new basin; resumable, a basin already in the file is skipped)
points = list(C.PLACES.values()) + [(float(s['lat']), float(s['lon'])) for s in SAMPLE]
t0 = time.perf_counter()
n_cores, n_new = R.fetch_table(points, conn)
print(f'ring table: {n_cores} basins in {R.RING_CSV.name} | {n_new} fetched this run | {time.perf_counter() - t0:.0f}s')

  25 points | 0 new basins | 0s
  50 points | 0 new basins | 0s
  75 points | 0 new basins | 0s
  100 points | 0 new basins | 0s
  125 points | 0 new basins | 0s
  150 points | 0 new basins | 0s
  175 points | 0 new basins | 0s
  200 points | 0 new basins | 0s
  225 points | 0 new basins | 0s
  250 points | 0 new basins | 0s
  275 points | 0 new basins | 0s
  300 points | 0 new basins | 0s
ring table: 315 basins in wo06_ring_values.csv | 0 fetched this run | 0s


In [3]:
# Cell 3 — The ring table against the saved WO-03 pair table: neighbour counts must agree
RING = R.load_table()
rows = []
for hid, d in RING.items():
    ring_nb = d[(d['role'] == 'neighbour') & (d['shared_km'] > 0)]['member'].nunique()
    pc = PAIRS[PAIRS['core_id'] == hid].drop_duplicates('nb_id')
    rows.append(dict(core=hid, ring_nb=ring_nb, saved_nb=int((pc['shared_km'] > 0).sum()), point_only=d[(d['role'] == 'neighbour') & (d['shared_km'] == 0)]['member'].nunique()))
chk = pd.DataFrame(rows)
bad = chk[chk['ring_nb'] != chk['saved_nb']]
L = [f'basins in the ring table: {len(chk)} | variables per basin: {pd.concat(RING.values())["variable"].nunique()}',
     f'border-neighbour count differs from the saved pair table at {len(bad)} basins',
     'neighbours per basin (border only): ' + str(chk['ring_nb'].value_counts().sort_index().to_dict()),
     f'fewer than {R.MIN_NB} neighbours (ring rules stay silent): {int((chk["ring_nb"] < R.MIN_NB).sum())} | basins with a point-only neighbour: {int((chk["point_only"] > 0).sum())}']
if len(bad):
    L.append(bad.head(10).to_string())
print('\n'.join(L))

basins in the ring table: 315 | variables per basin: 45
border-neighbour count differs from the saved pair table at 0 basins
neighbours per basin (border only): {0: 8, 1: 2, 2: 18, 3: 31, 4: 55, 5: 64, 6: 50, 7: 37, 8: 17, 9: 16, 10: 7, 11: 1, 12: 5, 14: 1, 15: 2, 18: 1}
fewer than 3 neighbours (ring rules stay silent): 28 | basins with a point-only neighbour: 23


In [4]:
# Cell 4 — The ten test places: facts, units, and the filled_profiles (written to docs/design/dashboard/wo06_filled_profiles.md)
OUTMD = ROOT / 'docs' / 'design' / 'dashboard' / 'wo06_filled_profiles.md'
t0 = time.perf_counter()
TEST = {}
parts = ['# WO-06: the dummy filled from real data, ten test places (filled_profiles)', '',
         'Generated by `notebooks/edop/whg/wo06_rings.ipynb` (commit ' + C.commit() + '). Every `yes` unit of `modal_dummy_annotated.md` is filled from data; each `maybe` or `no` unit is a placeholder in the form `\u27e6id maybe: needs ...\u27e7`. Ring comparison units (S2.3, S5.2, S6.2 to S6.5) are silent where they have nothing true to say. Wording is the template file\'s, not polished.', '',
         'Wording from `' + W.templates_stamp() + '`.', '']
for name, (lat, lon) in C.PLACES.items():
    F = W.build_facts(lat, lon, conn, PAIRS, FLOW, ring=RING)
    U = W.fill_units(F)
    TEST[name] = (F, U)
    parts.append(W.render_filled_profile(name, F, U))
    parts.append('---'); parts.append('')
OUTMD.write_text('\n'.join(parts))
print(f'wrote {OUTMD.name} | {W.templates_stamp()} | {time.perf_counter() - t0:.0f}s\n\n' + W.render_filled_profile('Tbilisi', *TEST['Tbilisi']))

wrote wo06_filled_profiles.md | profile_templates.md v5 (last commit 58fae48; edited since: yes); script wo04_fill.py (last commit fd6050d; edited since: yes); script wo06_rings.py (last commit fd6050d; edited since: yes) | 12s

## Tbilisi: environmental profile

### Setting
Tbilisi sits at about 418 m above sea level, in a basin of ⟦S1.1c maybe: needs terrain scheme or cutoff (wo04b)⟧ country that runs from about 255 m to 3,838 m. It lies in the Azerbaijan shrub desert and steppe ecoregion, ⟦S1.2b maybe: needs stored one-line ecoregion gist⟧. Most of its basin lies in the Caucasus mixed forests ecoregion. ⟦S1.3 maybe: needs distance from the point to the next ecoregion⟧ ⟦S1.4 maybe: needs share cutoff for further ecoregions⟧

### Climate
Across the surrounding basin of about 23,251 km², rainfall averages about 762 mm a year, ⟦S2.1b maybe: needs rule for "mostly in {months}"⟧. Temperatures there average 5.3 °C, from -5.9 °C in January to 15.8 °C in August. Its basin is cooler than seve

In [5]:
# Cell 5 — The sample: 306 basin interior points through the ring rules; saved to a CSV (about 8 min: the single-basin calls, not the ring)
rows, t_start = [], time.perf_counter()
for k, s in enumerate(SAMPLE, 1):
    F = W.build_facts(float(s['lat']), float(s['lon']), conn, PAIRS, FLOW, ring=RING)
    U = W.fill_units(F)
    d = dict(core_id=int(s['hybas_id_l6']), zone=s['climate_zone'], n_nb=F['n_nb'])
    for u in ('S2.3', 'S5.2', 'S6.2', 'S6.3', 'S6.4', 'S6.5'):
        d[u] = int(U[u]['ok']); d[u + '_na'] = int(U[u].get('na', False)); d[u + '_why'] = U[u]['why'] or ''
    tr = {u: U[u].get('trace', {}) for u in ('S2.3', 'S5.2', 'S6.3', 'S6.4', 'S6.5', 'S6.ring')}
    d['S2.3_kind'], d['S2.3_ext'], d['S2.3_used'] = tr['S2.3'].get('kind', ''), len(tr['S2.3'].get('ext', [])), len(tr['S2.3'].get('used', []))
    d['S5.2_kind'], d['S5.2_ext'], d['S5.2_used'] = tr['S5.2'].get('kind', ''), len(tr['S5.2'].get('ext', [])), len(tr['S5.2'].get('used', []))
    d['S6.3_across'], d['S6.3_hits'] = tr['S6.3'].get('across', 0), len(tr['S6.3'].get('hit', []))
    d['S6.3_first'] = tr['S6.3']['hit'][0][0] + ' ' + tr['S6.3']['hit'][0][1] if tr['S6.3'].get('hit') else ''
    d['S6.4_k'], d['S6.4_m'] = tr['S6.4'].get('k'), tr['S6.4'].get('m')
    d['S6.5_ext'], d['S6.5_used'] = len(tr['S6.5'].get('ext', [])), len(tr['S6.5'].get('used', []))
    d['core_smallest'], d['core_largest'] = tr['S6.ring'].get('smallest'), tr['S6.ring'].get('largest')
    d['text_S2.3'], d['text_S6.5'] = (U['S2.3']['text'] or ''), (U['S6.5']['text'] or '')
    rows.append(d)
    if k % 50 == 0:
        print(f'  {k} of {len(SAMPLE)} in {time.perf_counter() - t_start:.0f}s')
S5 = pd.DataFrame(rows)
S5.to_csv(C.OUT / 'wo06_fill.csv', index=False)
print(f'rows saved: {len(S5)} | this run took {time.perf_counter() - t_start:.0f}s')

rows saved: 306 | this run took 481s


In [6]:
# Cell 6 — Firing rates over the sample: how often each unit speaks, how many extremes a basin has, how often the cap bites, S6.3 and S6.4
L = [f'sample basins: {len(S5)}']
for u in ('S2.3', 'S5.2', 'S6.2', 'S6.3', 'S6.4', 'S6.5'):
    L.append(f'{u}: speaks {int(S5[u].sum())} | silent (nothing true to say) {int(S5[u + "_na"].sum())} | could not fill {int(((S5[u] == 0) & (S5[u + "_na"] == 0)).sum())}')
L.append('')
for u, cap in (('S2.3', R.CAP['S2_3']), ('S5.2', R.CAP['S5_2'])):
    L.append(f'{u}: kind {S5[u + "_kind"].value_counts().to_dict()} | extremes found per basin {S5[u + "_ext"].value_counts().sort_index().to_dict()} | cap {cap} bites at {int((S5[u + "_ext"] > cap).sum())}')
L.append(f'S6.5: extremes found per basin {S5["S6.5_ext"].value_counts().sort_index().to_dict()} | cap {R.CAP["S6_5"]} bites at {int((S5["S6.5_ext"] > R.CAP["S6_5"]).sum())}')
spoke = S5[S5['S6.3'] == 1]
L.append(f'S6.3 ("all across its divides ..."): speaks at {len(spoke)} | basins with a first measure: ' + str(spoke['S6.3_first'].value_counts().to_dict()))
# S6.4_k is the number of measures on which the core lies OUTSIDE its neighbours' range; S6.4 speaks only when k > 0
m4 = S5[S5['S6.4_m'] > 0]
out4 = m4[m4['S6.4_k'] > 0]
L.append(f'S6.4: measures compared per basin (min/median/max) {int(m4["S6.4_m"].min())}/{int(m4["S6.4_m"].median())}/{int(m4["S6.4_m"].max())} | within the range on every measure (silent) at {int((m4["S6.4_k"] == 0).sum())} of {len(m4)} | outside on k measures, where it speaks (min/median/max) {int(out4["S6.4_k"].min())}/{int(out4["S6.4_k"].median())}/{int(out4["S6.4_k"].max())} | k distribution {out4["S6.4_k"].value_counts().sort_index().to_dict()}')
L.append('')
S5['nb_band'] = pd.cut(S5['n_nb'], [-1, 2, 4, 6, 8, 99], labels=['0-2', '3-4', '5-6', '7-8', '9+'])
S5['S2.3_is_extreme'] = (S5['S2.3_kind'] == 'extreme').astype(int)
S5['S5.2_is_extreme'] = (S5['S5.2_kind'] == 'extreme').astype(int)
g = S5.groupby('nb_band', observed=True)[['S2.3_is_extreme', 'S5.2_is_extreme', 'S6.5', 'S6.3']].mean()
g.insert(0, 'basins', S5.groupby('nb_band', observed=True).size())
L.append('by number of neighbours (share of basins that get an extreme, or that speak):'); L.append(g.round(2).to_string())
S5['area_rank'] = np.where(S5['core_smallest'] == True, 'core smallest', np.where(S5['core_largest'] == True, 'core largest', 'neither'))
S5['any_extreme'] = ((S5['S2.3_ext'] + S5['S5.2_ext'] + S5['S6.5_ext']) > 0).astype(int)
S5['all_within'] = ((S5['S6.4_m'] > 0) & (S5['S6.4_k'] == 0)).astype(int)
big = S5[S5['n_nb'] >= R.MIN_NB]
h = big.groupby('area_rank')[['any_extreme', 'all_within']].mean()
h.insert(0, 'basins', big.groupby('area_rank').size())
L.append(''); L.append('by the core\'s size among its ring (share with at least one extreme; share within the range on every S6.4 measure, i.e. S6.4 silent):'); L.append(h.round(2).to_string())
print('\n'.join(L))

sample basins: 306
S2.3: speaks 278 | silent (nothing true to say) 28 | could not fill 0
S5.2: speaks 232 | silent (nothing true to say) 74 | could not fill 0
S6.2: speaks 298 | silent (nothing true to say) 8 | could not fill 0
S6.3: speaks 70 | silent (nothing true to say) 236 | could not fill 0
S6.4: speaks 208 | silent (nothing true to say) 98 | could not fill 0
S6.5: speaks 144 | silent (nothing true to say) 162 | could not fill 0

S2.3: kind {'count': 219, 'extreme': 59, '': 28} | extremes found per basin {0: 247, 1: 35, 2: 15, 3: 9} | cap 2 bites at 9
S5.2: kind {'count': 152, 'extreme': 80, 'none': 46, '': 28} | extremes found per basin {0: 226, 1: 62, 2: 14, 3: 4} | cap 2 bites at 4
S6.5: extremes found per basin {0: 162, 1: 69, 2: 34, 3: 18, 4: 13, 5: 7, 6: 2, 7: 1} | cap 3 bites at 23
S6.3 ("all across its divides ..."): speaks at 70 | basins with a first measure: {'elev less': 21, 'elev more': 17, 'precip less': 9, 'precip more': 7, 'temp less': 7, 'temp more': 5, 'aridity l

In [7]:
# Cell 7 — The ring sentences at the ten places, to read against what is known of each place
L = []
for name in C.PLACES:
    F, U = TEST[name]
    if F['pub'] is None:
        L.append(f'{name}: no basin'); continue
    sents = [(u, U[u]['text']) for u in ('S2.3', 'S5.2', 'S6.2', 'S6.3', 'S6.4', 'S6.5') if U.get(u, {}).get('ok')]
    silent = [u for u in ('S2.3', 'S5.2', 'S6.2', 'S6.3', 'S6.4', 'S6.5') if not U.get(u, {}).get('ok')]
    L.append(f'{name} ({F["n_nb"]} neighbours; {U.get("S6.ring", {}).get("note", "")})')
    L += [f'   {u}: {t}' for u, t in sents] + ['   silent: ' + (', '.join(silent) or 'none')]
print('\n'.join(L))

Timbuktu (5 neighbours; 5 neighbours; areas km2 core 3,826, neighbours 921 to 24,966; the core is neither the smallest nor the largest by area)
   S2.3: Its basin receives less precipitation than three of the five basins around it.
   S5.2: Its basin has the highest population density of the six basins that meet here.
   S6.2: Two of them drain into it. It drains into one of them.
   S6.4: On seven of 22 measures, its values fall outside the range of the basins around it.
   S6.5: Of the six basins that meet here, it has the lowest mean elevation, the least relief and the gentlest slopes.
   silent: S6.3
Tbilisi (9 neighbours; 9 neighbours; areas km2 core 23,251, neighbours 2,976 to 22,012; the core is the largest by area)
   S2.3: Its basin is cooler than seven of the nine basins around it.
   S5.2: Its basin has more cropland than seven of the nine basins around it.
   S6.2: None of them drains into it. It drains into one of them.
   silent: S6.3, S6.4, S6.5
Santa Fe (7 neighbours; 7